[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C58_HardCase_LongTail_Course/02_hard_mining/02_hard_example_mining.ipynb)

# 02 · 难例挖掘（OHEM / NMS 去重 / Focal 是软性 OHEM / 噪声标签陷阱 / AUM）

目标：把「难例挖掘」从一句口号变成可度量的工程。你会亲手看到
**不去重的 OHEM 为什么比不做 OHEM 更差**、**Focal Loss 与 OHEM 在数学上是同一件事的软硬两版**，
以及本模块最重要的那个实验——**挖掘过头会把噪声标签当难例，把模型毁掉**。

本 notebook 你会亲手实现：
1. **IoU / NMS / OHEM**，并量化「不去重时 top-k 被 2–4 个区域吃光」
2. 正负样本**分开挖**与 loss 归一化（除以 k 还是除以 N）
3. **Focal 权重与 OHEM 掩码的排序一致性证明** + 有效样本数 `n_eff=(Σw)²/Σw²` → γ ↔ 等价 k 换算
4. **噪声放大系数 A** 的测量：top-1% 高 loss 样本里噪声占比 / 总体噪声率
5. **关键实验**：干净 vs 20% 噪声 × {均匀采样, OHEM, Focal}，看清 OHEM 如何记住噪声、毁掉泛化
6. **AUM（损失轨迹）** 区分难例与噪声：证明「只看最终 loss 分不开，看整条轨迹分得开」
7. **多模型一致性**的二维判据 + 重标注抽检的样本量计算
8. **难例池**（EMA / 老化 / 过采样封顶 / 桶配额）与**安全挖掘比例**公式

> 心智模型：**难例挖掘不是「找最难的」，而是「在给定标注质量下，找信息量最大且可信的那批」。
> 「可信」这一半是绝大多数教程会漏掉的，而它在真实数据上决定成败。**

## 1 · 合成候选框：难例天然「扎堆」

真实检测器一张图上有 10³–10⁵ 个候选框，而且**一个难区域周围会有几十个高度重叠的候选**。
这个性质是下一节 NMS 去重的全部理由，所以先把它造出来。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def make_proposals(n_hot=8, per_hot=40, n_scatter=1600, W=640, H=384, seed=0):
    """合成一张图的候选框。
       · n_hot 个「难区域热点」，每个周围有 per_hot 个**高度重叠**的候选（loss 都很高）
       · n_scatter 个散落的简单背景候选（loss 接近 0）
       region id: 0..n_hot-1 表示热点；1000+ 表示各自独立的散点区域"""
    r = np.random.default_rng(seed)
    boxes, loss, region = [], [], []
    centers = r.uniform([80, 80], [W - 80, H - 80], size=(n_hot, 2))
    hot_loss = np.linspace(3.0, 2.0, n_hot)          # 热点之间难度拉开，便于观察 top-k 归属
    for i, (cx, cy) in enumerate(centers):
        for _ in range(per_hot):
            jx, jy = r.normal(0, 1.5, 2)             # 抖动很小 -> 彼此 IoU 很高
            s = 32.0
            boxes.append([cx + jx - s/2, cy + jy - s/2, cx + jx + s/2, cy + jy + s/2])
            loss.append(hot_loss[i] + r.normal(0, 0.03))
            region.append(i)
    for j in range(n_scatter):
        cx, cy = r.uniform([30, 30], [W - 30, H - 30])
        s = r.uniform(20, 60)
        boxes.append([cx - s/2, cy - s/2, cx + s/2, cy + s/2])
        loss.append(r.exponential(0.12))             # 简单背景：梯度 ~1e-3 量级
        region.append(1000 + j)
    return np.asarray(boxes), np.asarray(loss), np.asarray(region)

BOXES, LOSS, REGION = make_proposals()
easy = (LOSS < 0.5).mean()
print(f'候选框总数 {len(BOXES)}（8 个热点 × 40 个重叠候选 + 1600 个散点）')
print(f'loss 分位数  p50={np.percentile(LOSS,50):.3f}  p90={np.percentile(LOSS,90):.3f}  max={LOSS.max():.3f}')
print(f'loss < 0.5 的「简单样本」占比 {easy:.1%}   ← 它们的梯度 |p-y| ~ 1e-3，数量却占绝对多数')
assert easy > 0.78
assert LOSS[REGION < 1000].min() > LOSS[REGION >= 1000].max(), '热点区域的 loss 应全面高于散点'
print('✅ 数据就位：难例天然扎堆 —— 这正是朴素 OHEM 会失效的原因')

## 2 · OHEM：去重与不去重，差别是「有效 batch = 3」还是「= 100」

OHEM 的机制是「按 loss 排序取 top-k 反传」。
**但候选框高度重叠，直接排序会让 top-k 被一两个热点吃光。**
原论文的做法是：**先用 loss 当分数做一遍 NMS（IoU 0.7），再取 top-k**。

In [ ]:
def iou_1_to_n(box, others):
    x1 = np.maximum(box[0], others[:, 0]); y1 = np.maximum(box[1], others[:, 1])
    x2 = np.minimum(box[2], others[:, 2]); y2 = np.minimum(box[3], others[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a = (box[2] - box[0]) * (box[3] - box[1])
    b = (others[:, 2] - others[:, 0]) * (others[:, 3] - others[:, 1])
    return inter / (a + b - inter + 1e-9)

def nms(boxes, scores, iou_thr):
    """标准贪心 NMS。返回的下标已按 score 降序。"""
    order = np.argsort(-scores); keep = []
    while order.size:
        i = order[0]; keep.append(i)
        if order.size == 1:
            break
        ious = iou_1_to_n(boxes[i], boxes[order[1:]])
        order = order[1:][ious <= iou_thr]
    return np.asarray(keep, dtype=int)

def ohem(boxes, losses, k, dedup_iou=None):
    """dedup_iou=None -> 朴素 OHEM（**错误实现**）
       dedup_iou=0.7   -> 原论文实现：**先用 loss 当分数做 NMS**，再取 top-k"""
    if dedup_iou is None:
        return np.argsort(-losses)[:k]
    return nms(boxes, losses, dedup_iou)[:k]

K = 128
sel_naive = ohem(BOXES, LOSS, K)
sel_dedup = ohem(BOXES, LOSS, K, dedup_iou=0.7)

def cover(sel):
    reg = REGION[sel]
    uniq, cnt = np.unique(reg, return_counts=True)
    return len(uniq), int(cnt.max())

n_naive, mx_naive = cover(sel_naive)
n_dedup, mx_dedup = cover(sel_dedup)
print(f'{"实现":<26s}{"top-128 覆盖的不同区域数":>24s}{"单区域最多占":>14s}')
print(f'{"朴素 OHEM（不去重）":<24s}{n_naive:>22d}{mx_naive:>14d}')
print(f'{"OHEM + NMS 去重(0.7)":<24s}{n_dedup:>22d}{mx_dedup:>14d}')
assert n_naive <= 6,  '不去重时 top-k 应被极少数热点吃光'
assert n_dedup >= 60, '去重后 top-k 应覆盖大量不同区域'
assert mx_naive >= 30 and mx_dedup <= 12
print(f'\n⚠️  不去重：128 个样本其实是 {n_naive} 件事的 128 份拷贝 -> **有效 batch ≈ {n_naive}**')
print('    梯度被单个区域完全支配 -> loss 震荡、在几个区域上过拟合、换数据骤降。')
print('✅ 「用 loss 当分数做一遍 NMS」在论文里只是一句话，在代码里是决定成败的一行。')

In [ ]:
# ── 细节 ②：正负样本必须分开挖 ──
n_pos, n_hardneg, n_easyneg = 20, 80, 1820
loss_pos     = rng.gamma(2.0, 0.25, n_pos)          # 正样本：loss 中等
loss_hardneg = rng.uniform(2.0, 4.0, n_hardneg)     # 难负样本：高分假正例
loss_easyneg = rng.exponential(0.10, n_easyneg)     # 简单负样本
all_loss = np.r_[loss_pos, loss_hardneg, loss_easyneg]
is_pos   = np.r_[np.ones(n_pos, bool), np.zeros(n_hardneg + n_easyneg, bool)]

K2 = 64
top_mixed = np.argsort(-all_loss)[:K2]
print(f'不分开挖  -> top-{K2} 里正样本数 = {int(is_pos[top_mixed].sum())}   ← 回归分支拿不到任何梯度')
assert is_pos[top_mixed].sum() == 0

kp = K2 // 4; kn = K2 - kp                           # 维持 1:3 正负比
idx_p = np.where(is_pos)[0];  idx_n = np.where(~is_pos)[0]
sel_p = idx_p[np.argsort(-all_loss[idx_p])[:kp]]
sel_n = idx_n[np.argsort(-all_loss[idx_n])[:kn]]
print(f'正负分开挖 -> 正 {len(sel_p)} 个 + 负 {len(sel_n)} 个（1:3）')
assert len(sel_p) == kp and is_pos[sel_p].all()

# ── 细节 ③：loss 除以 k 还是除以 N ──
sel = np.r_[sel_p, sel_n]
g_over_k = all_loss[sel].sum() / len(sel)
g_over_N = all_loss[sel].sum() / len(all_loss)
print(f'\n除以 k: {g_over_k:.3f}   除以 N: {g_over_N:.4f}   比值 {g_over_k/g_over_N:.1f}×')
print(f'⚠️  除以 N 相当于把学习率悄悄乘上 k/N = {len(sel)}/{len(all_loss)} = {len(sel)/len(all_loss):.3f}')
assert abs(g_over_k / g_over_N - len(all_loss) / len(sel)) < 1e-6
print('✅ 正确做法：除以**实际参与反传的样本数 k**，这样 k 才是一个可独立调的超参。')

## 3 · Focal Loss 是软性 OHEM

两者都能写成 $\mathcal{L}=\sum_i w_i\ell_i$：
OHEM 的 $w_i=\frac{1}{k}\mathbb{1}[\ell_i\ge\ell_{(k)}]$（**阶跃**），
Focal 的 $w_i=(1-p_i)^\gamma$（**连续**）。
下面证明两件事：**① 排序完全一致**（所以 focal 的 top-k 就是 OHEM 会选的那 k 个）；
**② 用有效样本数把 γ 翻译成「等价的 k」**。

In [ ]:
p = rng.uniform(0.005, 0.995, 4000)          # 模型给**正确类别**的概率 p_t
ce = -np.log(p)                                 # 交叉熵 loss
focal_w = lambda pt, g: (1.0 - pt) ** g

# ① 排序一致性：CE loss 与 focal 权重都是 p_t 的严格单调减函数
order_ce = np.argsort(-ce)
for g in [0.5, 1.0, 2.0, 5.0]:
    assert np.array_equal(np.argsort(-focal_w(p, g)), order_ce), f'gamma={g}'
print('✅ 对任意 γ，「按 focal 权重排序」与「按 loss 排序」**完全一致**')
print('   ⇒ focal 眼中最重要的 k 个样本，就是 OHEM 会选中的那 k 个。')
print('   ⇒ 二者唯一的差别：**被排除的样本，权重是恰好 0（OHEM）还是很小的 ε（focal）**。\n')

# ② 两种权重的形状对比
print(f'{"p_t":>7s}{"CE loss":>10s}{"focal w (γ=2)":>15s}{"OHEM w (top-10%)":>19s}')
thr = np.percentile(ce, 90)
for pt in [0.99, 0.9, 0.7, 0.5, 0.3, 0.1, 0.02]:
    l = -np.log(pt)
    print(f'{pt:>7.2f}{l:>10.3f}{(1-pt)**2:>15.4f}{("1.0" if l >= thr else "0.0（丢弃）"):>19s}')
print('\n⚠️  OHEM 把 p_t=0.7 这类「还没学好但也不算太差」的样本权重直接置 0；')
print('    focal 给它 0.09 的权重 —— **保留但降权**。这是 OHEM「会忘掉已学会的东西」的根源。')

In [ ]:
# ── 有效样本数：把 γ 翻译成「等价的 top-k」──
def n_eff(w):
    """加权平均的等效样本量 (Σw)²/Σw²。
       · 对 OHEM（k 个权重为 1，其余为 0）恰好等于 k —— 所以这个量可以直接对齐两种方法"""
    w = np.asarray(w, dtype=float)
    return float(w.sum() ** 2 / (w ** 2).sum())

# 先验证它在 OHEM 上就是 k
w_ohem = np.zeros(1000); w_ohem[:137] = 1.0
assert abs(n_eff(w_ohem) - 137) < 1e-9
print(f'n_eff(OHEM top-137) = {n_eff(w_ohem):.1f}  ✅ 与 k 完全一致\n')

# 一个真实感的 p_t 分布：绝大多数样本已经学得很好
p_real = rng.beta(9.0, 1.0, 20000)
N = len(p_real)
print(f'{"γ":>5s}{"n_eff":>12s}{"占全体比例":>14s}{"等价的 OHEM k":>16s}')
prev = None
for g in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]:
    ne = n_eff(focal_w(p_real, g))
    print(f'{g:>5.1f}{ne:>12.1f}{ne/N:>13.1%}{int(round(ne)):>16d}')
    if prev is not None:
        assert ne < prev, 'γ 越大，等效样本量越小'
    prev = ne
assert abs(n_eff(focal_w(p_real, 0.0)) - N) < 1e-6, 'γ=0 时 focal 退化为 CE，n_eff 应等于 N'
print('\n✅ γ 不是一个玄学超参：它等价于「只让 n_eff 个样本真正贡献梯度」。')
print('⚠️  这也解释了为什么 γ 从 2 调到 5 常常直接训崩 —— 等效样本量掉一个数量级，梯度方差爆炸。')
print('⚠️  推论：**OHEM 与 Focal 不要同时开**，否则等效样本量被平方级压缩。')

## 4 · 难例的类型学：loss 排序分不清「难」和「错」

四类样本——① 相似类混淆 ② 背景误检 ③ 边界样本 ④ **标注错误**——的 loss 分布是**重叠**的，
而且 ④ 系统性地排在最前面。下面用一个 AUC 量化「单靠 loss 能不能把 ④ 从 ③ 里分出来」。

In [ ]:
def auc(scores_pos, scores_neg):
    """Mann-Whitney U 形式的 AUC：P(score_pos > score_neg)，含并列各算一半。"""
    s = np.r_[scores_pos, scores_neg]
    r = np.empty(len(s)); order = np.argsort(s, kind='mergesort')
    sr = s[order]; i = 0
    while i < len(s):                                  # 处理并列：取平均秩
        j = i
        while j + 1 < len(s) and sr[j + 1] == sr[i]:
            j += 1
        r[order[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    n1 = len(scores_pos)
    return (r[:n1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * len(scores_neg))

assert abs(auc([3., 2.], [1., 0.]) - 1.0) < 1e-12
assert abs(auc([1., 1.], [1., 1.]) - 0.5) < 1e-12

GROUPS = {
    '① 相似类混淆': rng.gamma(3.0, 0.28, 600),     # 中等 loss
    '② 背景误检':   rng.gamma(2.0, 0.55, 400),     # 分布很宽
    '③ 边界样本':   rng.gamma(6.0, 0.32, 500),     # 偏高
    '④ 标注错误':   rng.gamma(8.0, 0.32, 120),     # **系统性最高**
}
print(f'{"类型":<16s}{"n":>6s}{"loss 均值":>11s}{"loss p90":>10s}')
for kname, v in GROUPS.items():
    print(f'{kname:<14s}{len(v):>6d}{v.mean():>11.2f}{np.percentile(v,90):>10.2f}')

a_noise_vs_hard = auc(GROUPS['④ 标注错误'], GROUPS['③ 边界样本'])
print(f'\n只用 loss 区分「④ 标注错误」vs「③ 边界样本」的 AUC = {a_noise_vs_hard:.3f}')
assert 0.55 < a_noise_vs_hard < 0.85, 'loss 有一点信号，但远不足以判别'
top1 = np.argsort(-np.concatenate(list(GROUPS.values())))[:int(0.01 * sum(map(len, GROUPS.values())))]
labels = np.concatenate([[k] * len(v) for k, v in GROUPS.items()])
frac4 = (labels[top1] == '④ 标注错误').mean()
enrich = frac4 / (120 / 1620)
print(f'top-1% 最高 loss 里「④ 标注错误」占 {frac4:.0%}（全体里只占 {120/1620:.0%}）-> 富集 {enrich:.1f}×')
assert enrich > 2.5, '挖得越狠，选中样本里标注错误的占比越高'
print('\n⚠️  loss 有信号但**不足以判别**（AUC 只有 0.6-0.8）——')
print('    而挖掘越激进，选中的样本里「④ 标注错误」的占比越高。这就是下一节的主题。')

## 5 · 挖掘过头：噪声标签会被当成最有价值的难例

先量化**噪声放大系数** $A=\dfrac{P(\text{noisy}\mid\text{selected})}{P(\text{noisy})}$，
再做本模块最重要的实验：**同一份数据、同一个模型，只换采样策略**，看 OHEM 如何在 20% 噪声下把模型毁掉。

In [ ]:
# ── 噪声放大系数 A ──
Nn, q = 5000, 0.08
noisy_flag = rng.random(Nn) < q
loss_mix = np.where(noisy_flag,
                    rng.gamma(6.0, 0.50, Nn),      # 噪声样本：与任何可学规律矛盾 -> loss 长期最高
                    rng.gamma(1.5, 0.35, Nn))      # 干净样本
base_rate = noisy_flag.mean()
print(f'总体噪声率 q = {base_rate:.1%}')
print(f'{"挖掘比例 r":>11s}{"选中样本里的噪声率":>20s}{"放大系数 A":>13s}')
As = {}
for frac in [0.01, 0.02, 0.05, 0.10, 0.30, 1.00]:
    kk = max(1, int(Nn * frac))
    sel = np.argsort(-loss_mix)[:kk]
    rate = noisy_flag[sel].mean()
    As[frac] = rate / base_rate
    print(f'{frac:>11.0%}{rate:>19.1%}{As[frac]:>13.2f}×')
assert As[0.01] > 5.0,  'top-1% 里噪声应被强烈富集'
assert As[0.01] > As[0.10] > As[1.00], '挖得越狠，噪声富集越严重'
assert abs(As[1.00] - 1.0) < 1e-9, '全选 = 不挖掘 -> A=1'
print('\n⚠️  A 是「难例挖掘的噪声放大器增益」。q=8% 的数据，挖 top-1% 时')
print(f'    实际喂给模型的噪声率高达 {base_rate*As[0.01]:.0%} —— 而你以为自己在喂难例。')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 关键实验：干净 vs 20% 噪声  ×  {均匀采样, OHEM, Focal}
# 同一份数据、同一个模型（2-32-1 的 tanh MLP）、同样的步数，**只换采样策略**
# ══════════════════════════════════════════════════════════════
WARMUP, STEPS, KSEL = 100, 1000, 80

def make_toy(n=800, noise=0.0, seed=1, sep=2.0):
    r = np.random.default_rng(seed)
    y = (r.random(n) < 0.5).astype(float)
    mu = np.where(y[:, None] > 0.5, np.array([sep, 0.0]), np.array([-sep, 0.0]))
    X = mu + r.normal(0, 1.0, size=(n, 2))
    y_obs = y.copy()
    flip = r.random(n) < noise                       # **标签噪声**：随机翻转
    y_obs[flip] = 1.0 - y_obs[flip]
    return X, y, y_obs, flip

def mlp_init(h=32, seed=0):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, 0.8, (2, h)), 'b1': np.zeros(h),
            'W2': r.normal(0, 0.8, (h, 1)), 'b2': np.zeros(1)}

def mlp_prob(par, X):
    H = np.tanh(X @ par['W1'] + par['b1'])
    return 1.0 / (1.0 + np.exp(-(H @ par['W2'] + par['b2']).ravel())), H

def fit(X, y, strategy='uniform', k=KSEL, gamma=2.0, steps=STEPS, warmup=WARMUP, lr=0.6, seed=0):
    par = mlp_init(seed=seed)
    hist = []
    for t in range(steps):
        p, H = mlp_prob(par, X)
        ell = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
        if t < warmup or strategy == 'uniform':      # **warmup 期一律均匀采样**
            sw = np.ones(len(y))
        elif strategy == 'ohem':
            sw = np.zeros(len(y)); sw[np.argsort(-ell)[:k]] = 1.0
        else:                                        # focal
            pt = np.where(y > 0.5, p, 1 - p); sw = (1 - pt) ** gamma
        sw = sw / (sw.sum() + 1e-12)                 # 归一化：三种策略的总步长可比
        hist.append(sw)
        d = (p - y) * sw
        gW2 = H.T @ d[:, None]; gb2 = np.array([d.sum()])
        dH = (d[:, None] @ par['W2'].T) * (1 - H ** 2)
        par['W1'] -= lr * (X.T @ dH); par['b1'] -= lr * dH.sum(0)
        par['W2'] -= lr * gW2;        par['b2'] -= lr * gb2
    return par, np.asarray(hist)

Xn, _, yn_obs, flip = make_toy(800, noise=0.20, seed=1)        # 20% 标签噪声
Xc, _, yc_obs, _    = make_toy(800, noise=0.00, seed=1)        # 同分布的干净版
Xte, yte, _, _      = make_toy(4000, noise=0.0, seed=99)       # **干净测试集**

rows = []
for tag, (Xd, yd, fl) in {'干净数据 (q=0%)':  (Xc, yc_obs, np.zeros(800, bool)),
                          '含噪数据 (q=20%)': (Xn, yn_obs, flip)}.items():
    for strat in ['uniform', 'ohem', 'focal']:
        par, sws = fit(Xd, yd, strategy=strat, seed=0)
        p_te, _ = mlp_prob(par, Xte)
        acc = float(((p_te > 0.5).astype(float) == yte).mean())      # 干净测试集上的泛化
        p_tr, _ = mlp_prob(par, Xd)
        memo = float(((p_tr[fl] > 0.5).astype(float) == yd[fl]).mean()) if fl.any() else float('nan')
        # 刚开启挖掘后 100 步里，分给噪声样本的梯度预算
        bud = float(sws[WARMUP:WARMUP + 100][:, fl].sum(axis=1).mean()) if fl.any() else float('nan')
        rows.append((tag, strat, acc, memo, bud))

print(f'{"数据":<18s}{"策略":<10s}{"干净测试集准确率":>18s}{"记住噪声标签":>15s}{"梯度预算给噪声":>17s}')
for tag, s, a, mm, bg in rows:
    f = lambda v: '—' if v != v else f'{v:.1%}'
    print(f'{tag:<16s}{s:<10s}{a:>17.1%}{f(mm):>16s}{f(bg):>17s}')

d = {(t, s): (a, mm, bg) for t, s, a, mm, bg in rows}
acc_cu, acc_co = d[('干净数据 (q=0%)', 'uniform')][0], d[('干净数据 (q=0%)', 'ohem')][0]
acc_u, acc_o, acc_f = (d[('含噪数据 (q=20%)', s)][0] for s in ['uniform', 'ohem', 'focal'])
memo_u, memo_o = (d[('含噪数据 (q=20%)', s)][1] for s in ['uniform', 'ohem'])
bud_u, bud_o = (d[('含噪数据 (q=20%)', s)][2] for s in ['uniform', 'ohem'])

assert acc_co > acc_cu - 0.03, '**干净数据上 OHEM 完全没问题** —— 这是对照组'
assert bud_o > 1.5 * bud_u,    'OHEM 把远超基线比例的梯度预算交给了噪声样本'
assert memo_o > memo_u + 0.15, 'OHEM 显著加速了对噪声标签的**记忆化**'
assert acc_o < acc_u - 0.15,   'OHEM 在含噪数据上的泛化崩坏'
assert acc_f > acc_u - 0.03,   'Focal 的软加权对噪声鲁棒得多'
print(f'\n✅ 对照组：**干净数据上 OHEM 一点问题都没有**（{acc_co:.1%} vs 均匀 {acc_cu:.1%}）。')
print(f'⚠️  换成 20% 噪声：OHEM 把 {bud_o:.0%} 的梯度预算给了噪声样本（基线只有 {bud_u:.0%}，'
      f'放大 {bud_o/bud_u:.1f}×），')
print(f'    对噪声标签的记忆率从 {memo_u:.0%} 飙到 {memo_o:.0%}，')
print(f'    干净测试集准确率从 {acc_u:.1%} **崩到 {acc_o:.1%}** —— 模型被自己的挖掘策略毁掉了。')
print(f'✅ Focal（软加权）{acc_f:.1%}，几乎不受影响 —— 权重上界是 1，且大量简单样本仍在分母里稀释。')
print('✅ 结论：**数据越脏，越不能硬挖。**「挖多狠」由标注质量决定，不由直觉决定。')

## 6 · 用损失轨迹（AUM）区分难例与噪声

原理是**记忆化效应的时间不对称性**：干净样本被「学会」（早期就快速下降），
噪声样本被「记住」（**晚得多、也突然得多**）。

$\text{AUM}_i=\frac{1}{E}\sum_t\big[z_{i,y_i}(t)-\max_{c\ne y_i}z_{i,c}(t)\big]$

下面证明关键的一点：**只看最终 margin 分不开，看整条轨迹分得开。**

In [ ]:
E = 60
tt = np.arange(E) / (E - 1)

def sim_traj(kind, n, r):
    """模拟 per-sample 的 margin 轨迹（被指派标签的 logit − 最大其他 logit）。"""
    if kind == 'easy':      base = -0.5 + 4.5 * (1 - np.exp(-5.0 * tt))
    elif kind == 'hard':    base = -0.8 + 1.8 * (1 - np.exp(-2.5 * tt))
    else:                   # noisy：长期为负（与标签矛盾），只在**末期**被硬记住
        base = -2.2 + 3.2 / (1 + np.exp(-(tt - 0.82) / 0.05))
    offset = r.normal(0, 0.50, (n, 1))              # 样本间差异
    return base[None, :] + offset + r.normal(0, 0.35, (n, E))

r2 = np.random.default_rng(7)
M = {'easy': sim_traj('easy', 1500, r2),
     'hard': sim_traj('hard', 300, r2),
     'noisy': sim_traj('noisy', 200, r2)}

aum   = {kk: v.mean(axis=1) for kk, v in M.items()}      # AUM = 全程 margin 的均值
final = {kk: v[:, -1]       for kk, v in M.items()}      # 只看最后一个 epoch

print(f'{"轨迹片段 (epoch)":<18s}' + ''.join(f'{int(e):>8d}' for e in [0, 10, 25, 40, 50, 59]))
for kk in ['easy', 'hard', 'noisy']:
    print(f'{kk:<18s}' + ''.join(f'{M[kk][:, e].mean():>8.2f}' for e in [0, 10, 25, 40, 50, 59]))

auc_aum   = auc(aum['hard'],   aum['noisy'])         # 越高说明越能把 hard 排在 noisy 之上
auc_final = auc(final['hard'], final['noisy'])
print(f'\n用 **AUM（整条轨迹均值）** 区分 hard vs noisy 的 AUC = {auc_aum:.3f}')
print(f'用 **最终 margin（只看最后一轮）** 的 AUC        = {auc_final:.3f}   ← 几乎等于瞎猜')
assert auc_aum > 0.90, 'AUM 应能很好地分开难例与噪声'
assert auc_final < 0.70, '最终 margin 分不开（因为噪声样本最终也被记住了）'

thr_aum = np.percentile(np.concatenate(list(aum.values())), 10)   # 取最低 10% 作可疑池
susp = {kk: (v < thr_aum).mean() for kk, v in aum.items()}
print(f'\nAUM 最低 10% 作为「可疑池」时的命中率：'
      f"noisy {susp['noisy']:.0%} | hard {susp['hard']:.0%} | easy {susp['easy']:.0%}")
assert susp['noisy'] > 0.7 and susp['easy'] < 0.05
print('\n✅ 工程成本几乎为零：训练时每 epoch 记一个 per-sample margin，100 万样本 × 50 epoch = 200 MB。')
print('⚠️  但注意 AUM 只给「怀疑」，不给「真值」—— 真值只能靠重标注（下一节）。')

## 7 · 多模型一致性：最锋利的二维判据 + 重标注抽检的样本量

判据：**K 个独立模型高度一致地预测了另一个类** → 强烈怀疑标注错；
**模型之间互相不一致** → 真难例。

In [ ]:
K_MODEL, N_CLS = 5, 8
r3 = np.random.default_rng(11)

def sim_votes(kind, n, label=0):
    """K 个独立模型对同一样本的预测类别。"""
    v = np.empty((n, K_MODEL), dtype=int)
    for i in range(n):
        for m in range(K_MODEL):
            u = r3.random()
            if kind == 'easy':                       # 模型一致且同意标签
                v[i, m] = label if u < 0.97 else r3.integers(1, N_CLS)
            elif kind == 'hard':                     # **模型互相吵架**
                v[i, m] = label if u < 0.45 else (1 if u < 0.725 else 2)
            else:                                    # noisy：模型一致地指向**真类别**(=3)
                v[i, m] = 3 if u < 0.90 else label
    return v

V = {'easy': sim_votes('easy', 1500), 'hard': sim_votes('hard', 300),
     'noisy': sim_votes('noisy', 200)}
LABEL = 0

def vote_stats(votes):
    maj, frac = [], []
    for row in votes:
        c = np.bincount(row, minlength=N_CLS)
        maj.append(int(c.argmax())); frac.append(c.max() / len(row))
    return np.asarray(maj), np.asarray(frac)

def mislabel_flag(votes, label, consensus_thr=0.8):
    maj, frac = vote_stats(votes)
    return (maj != label) & (frac >= consensus_thr)

print(f'{"类型":<10s}{"多数票占比均值":>16s}{"多数票≠标签的比例":>20s}{"被判「疑似标错」":>18s}')
flags = {}
for kk in ['easy', 'hard', 'noisy']:
    maj, frac = vote_stats(V[kk])
    flags[kk] = mislabel_flag(V[kk], LABEL)
    print(f'{kk:<10s}{frac.mean():>15.2f}{(maj != LABEL).mean():>19.0%}{flags[kk].mean():>18.0%}')

tp = int(flags['noisy'].sum())
fp = int(flags['hard'].sum() + flags['easy'].sum())
prec = tp / max(tp + fp, 1); rec = tp / len(flags['noisy'])
print(f'\n判据 (多数票≠标签 且 一致率≥0.8):  精确率 {prec:.1%}   召回率 {rec:.1%}')
assert rec > 0.80 and prec > 0.80
assert flags['hard'].mean() < 0.15, '真难例（模型互相吵架）不应被误判为标注错误'

# ── 重标注抽检需要多少样本？ ──
def n_for_ci(p, eps, z=1.96):
    return int(np.ceil(z ** 2 * p * (1 - p) / eps ** 2))
print(f'\n{"待估噪声率 p":>14s}{"目标精度 ±ε":>14s}{"需要抽检样本数":>16s}')
for pp, ee in [(0.10, 0.02), (0.10, 0.01), (0.05, 0.02), (0.20, 0.02)]:
    print(f'{pp:>14.0%}{ee:>14.1%}{n_for_ci(pp, ee):>16d}')
assert n_for_ci(0.10, 0.02) == 865
print('\n✅ 把噪声率估到 ±2% 只需 ~865 个双标样本 —— 完全负担得起，')
print('   而它换来的是整条挖掘链路的可信度（用来给 AUM / 一致性判据定阈值）。')
print('⚠️  同时要汇报 Cohen\'s κ：κ < 0.6 说明**标注规范本身有问题**，此时讨论噪声率没有意义。')

## 8 · 难例池与安全挖掘比例

真实系统不会每步重扫全量数据，而是维护一个**难例池**：EMA loss、老化、过采样封顶、按失效模式分桶。
挖掘比例 $r$ 则由标注质量决定：

$q_{\text{eff}}=(1-r)q+rqA\le q_{\max}\;\Longleftrightarrow\; r\le\dfrac{q_{\max}-q}{q(A-1)}$

In [ ]:
class HardPool:
    """难例池：EMA loss + 老化 + 过采样封顶 + 按失效模式分桶配额 + AUM 剔除。"""
    def __init__(self, decay=0.92, cap=5, max_age=8, bucket_quota=0.30):
        self.d = {}                      # sid -> dict
        self.decay, self.cap = decay, cap
        self.max_age, self.bucket_quota = max_age, bucket_quota

    def update(self, sids, losses, buckets, aums, epoch):
        for s, l, b, a in zip(sids, losses, buckets, aums):
            e = self.d.setdefault(s, {'ema': l, 'n': 0, 'bucket': b, 'aum': a, 'seen': epoch})
            e['ema'] = self.decay * e['ema'] + (1 - self.decay) * l
            e['seen'] = epoch; e['aum'] = a

    def age(self, epoch):
        """老化：踢掉长期没被重新评估的样本，避免池变成「化石集合」。"""
        drop = [s for s, e in self.d.items() if epoch - e['seen'] > self.max_age]
        for s in drop:
            del self.d[s]
        return len(drop)

    def sample(self, m, aum_thr=-np.inf, r=None):
        r = r or np.random.default_rng(0)
        cand = [(s, e) for s, e in self.d.items()
                if e['n'] < self.cap and e['aum'] >= aum_thr]     # 封顶 + 剔除疑似标错
        out, per_bucket = [], {}
        quota = max(1, int(self.bucket_quota * m))
        for s, e in sorted(cand, key=lambda kv: -kv[1]['ema']):
            b = e['bucket']
            if per_bucket.get(b, 0) >= quota:                     # **桶配额**
                continue
            per_bucket[b] = per_bucket.get(b, 0) + 1
            e['n'] += 1; out.append(s)
            if len(out) >= m:
                break
        return out, per_bucket

r4 = np.random.default_rng(3)
pool = HardPool()
BUCKETS = ['night', 'occlusion', 'similar_class', 'small']
sids = np.arange(600)
# 「夜间」桶天然 loss 更高 —— 不设配额就会吃光全部预算
bk = [BUCKETS[i % 4] for i in sids]
ls = np.array([2.6 if b == 'night' else r4.gamma(2.0, 0.35) for b in bk])
au = np.where(r4.random(600) < 0.05, -1.5, r4.normal(0.8, 0.4, 600))   # 5% 疑似标错
pool.update(sids, ls, bk, au, epoch=0)

sel_all, dist = pool.sample(60, r=r4)
print('不剔除疑似标错时，各桶占比:', {k: f'{v/60:.0%}' for k, v in dist.items()})
assert max(dist.values()) <= max(1, int(0.30 * 60)), '桶配额生效：单桶不超过 30%'

pool2 = HardPool(); pool2.update(sids, ls, bk, au, epoch=0)
sel_ok, _ = pool2.sample(60, aum_thr=0.0, r=r4)
bad_in = sum(1 for s in sel_ok if au[s] < 0)
print(f'剔除 AUM<0 后，选中样本里疑似标错的个数 = {bad_in}')
assert bad_in == 0
dropped = pool2.age(epoch=20)          # 池里样本上次评估是 epoch 0，max_age=8 -> 全部老化出局
print(f'老化：把 epoch 推到 20 后被踢出池的样本数 = {dropped}')
assert dropped == 600 and len(pool2.d) == 0, '不做老化的池会变成「化石集合」'

# ── 安全挖掘比例 ──
def safe_mining_ratio(q, A, q_max):
    """q: 标注噪声率; A: 噪声放大系数; q_max: 可容忍的有效噪声率。"""
    if A <= 1.0:
        return 1.0
    if q >= q_max:
        return 0.0
    return float(min(1.0, max(0.0, (q_max - q) / (q * (A - 1.0)))))

print(f'\n{"噪声率 q":>10s}{"放大系数 A":>13s}{"容忍上限 q_max":>16s}{"安全挖掘比例 r*":>17s}')
for qq, AA, qm in [(0.02, 6.0, 0.15), (0.05, 6.0, 0.15), (0.08, 6.0, 0.15),
                   (0.08, 10.0, 0.15), (0.15, 6.0, 0.15)]:
    print(f'{qq:>10.0%}{AA:>13.1f}{qm:>16.0%}{safe_mining_ratio(qq, AA, qm):>17.1%}')
assert abs(safe_mining_ratio(0.08, 6.0, 0.15) - 0.175) < 1e-9
assert safe_mining_ratio(0.15, 6.0, 0.15) == 0.0
assert safe_mining_ratio(0.02, 1.0, 0.15) == 1.0
print('\n✅ 「挖多狠」不是玄学：q=2% 的干净数据可以挖到 100%，q=8% 就只能挖 17.5%，')
print('   q≥q_max 时**一点都不能挖**——此时该做的是治理标注质量，不是调采样器。')

## ✏️ 练习 1：带去重的 OHEM

实现 `ohem_select(boxes, losses, k, iou_thr, min_loss=0.0)`：

1. 先丢掉 `losses < min_loss` 的样本（简单样本没必要进排序）
2. **用 loss 作为分数做一遍 NMS**（阈值 `iou_thr`），去掉空间重复
3. 返回 loss 最高的 k 个的**原始下标**（按 loss 降序）

In [ ]:
def ohem_select(boxes, losses, k, iou_thr=0.7, min_loss=0.0):
    # TODO: ① 按 min_loss 过滤 ② 用 loss 当分数做 NMS ③ 取 top-k，返回**原始下标**
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s = ohem_select(BOXES, LOSS, 128, iou_thr=0.7)
assert len(s) == 128 and len(set(s.tolist())) == 128, '不能有重复下标'
assert np.all(np.diff(LOSS[s]) <= 1e-12), '必须按 loss 降序返回'
nreg = len(np.unique(REGION[s]))
assert nreg >= 60, f'去重后应覆盖大量不同区域，实际 {nreg}'
assert np.array_equal(s, ohem(BOXES, LOSS, 128, dedup_iou=0.7)), '应与参考实现一致'
s2 = ohem_select(BOXES, LOSS, 10**9, iou_thr=0.7, min_loss=1.5)
assert LOSS[s2].min() >= 1.5, 'min_loss 过滤未生效'
assert len(s2) <= 40, '只有 8 个热点区域，去重后剩余应很少'
print(f'top-128 覆盖 {nreg} 个不同区域；min_loss=1.5 且去重后只剩 {len(s2)} 个候选')
print('✅ 练习 1 通过：**先 NMS 再 top-k** —— OHEM 的成败就在这一行')

## ✏️ 练习 2：Focal ↔ OHEM 的换算

实现两个函数：
- `focal_weights(p_t, gamma, alpha=None)`：返回 `(1-p_t)^gamma`；若给了 `alpha` 则再乘上 `alpha`
- `equivalent_k(p_t, gamma)`：返回该 γ 下的**有效样本数**（四舍五入取整），即「等价的 OHEM k」

In [ ]:
def focal_weights(p_t, gamma, alpha=None):
    # TODO
    raise NotImplementedError

def equivalent_k(p_t, gamma):
    # TODO: 用 n_eff(w) = (Σw)² / Σw²
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pt = rng.beta(9.0, 1.0, 20000)
assert np.allclose(focal_weights(np.array([0.5]), 2.0), np.array([0.25]))
assert np.allclose(focal_weights(np.array([0.5]), 2.0, alpha=0.25), np.array([0.0625]))
assert equivalent_k(pt, 0.0) == len(pt), 'γ=0 退化为 CE，等价 k = N'
ks = [equivalent_k(pt, g) for g in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]]
assert all(a > b for a, b in zip(ks, ks[1:])), 'γ 越大等价 k 越小'
assert equivalent_k(pt, 2.0) < 0.5 * len(pt)
# alpha 是常数缩放，**不改变有效样本数**（这是 α 与 γ 分工不同的数学体现）
w1, w2 = focal_weights(pt, 2.0), focal_weights(pt, 2.0, alpha=0.25)
assert abs(n_eff(w1) - n_eff(w2)) < 1e-6, 'α 只缩放整体幅值，不改变权重分布形状'
for g, kk in zip([0.0, 0.5, 1.0, 2.0, 3.0, 5.0], ks):
    print(f'γ={g:<4.1f} -> 等价 OHEM k = {kk:>6d}  ({kk/len(pt):.1%} 的样本在真正贡献梯度)')
print('✅ 练习 2 通过：γ 不是玄学 —— 它等价于一个 top-k，且 α 不参与这件事')

## ✏️ 练习 3：区分难例与噪声标签

实现 `triage(margins, votes, labels, susp_pct=10.0, consensus_thr=0.8)`，
对每个样本返回三种标签之一：

- `'mislabeled'`：AUM 在最低 `susp_pct` 百分位 **且** 多模型一致地投给了另一个类
- `'hard'`：AUM 在最低 `susp_pct` 百分位，但模型之间**不一致**（一致率 < `consensus_thr`）
- `'ok'`：其余

`margins` 形状 `(N, E)`，`votes` 形状 `(N, K)`，`labels` 形状 `(N,)`。

In [ ]:
def triage(margins, votes, labels, susp_pct=10.0, consensus_thr=0.8):
    # TODO: ① AUM = margins.mean(axis=1) ② 低分位阈值 ③ 多数票与一致率 ④ 三分类
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Mall = np.vstack([M['easy'], M['hard'], M['noisy']])
Vall = np.vstack([V['easy'], V['hard'], V['noisy']])
Lall = np.zeros(len(Mall), dtype=int)
truth = np.array(['easy'] * 1500 + ['hard'] * 300 + ['noisy'] * 200)
tag = triage(Mall, Vall, Lall, susp_pct=15.0)
assert set(np.unique(tag)) <= {'mislabeled', 'hard', 'ok'}
rec_mis = (tag[truth == 'noisy'] == 'mislabeled').mean()
fp_easy = (tag[truth == 'easy'] == 'mislabeled').mean()
rec_hard = (tag[truth == 'hard'] == 'hard').mean()
print(f'{"真值":<10s}{"判为 mislabeled":>18s}{"判为 hard":>12s}{"判为 ok":>10s}')
for t in ['easy', 'hard', 'noisy']:
    sub = tag[truth == t]
    print(f'{t:<10s}{(sub=="mislabeled").mean():>17.0%}{(sub=="hard").mean():>12.0%}{(sub=="ok").mean():>10.0%}')
assert rec_mis > 0.60, f'噪声样本召回过低: {rec_mis:.2f}'
assert fp_easy < 0.03, f'简单样本不应被判为标注错误: {fp_easy:.3f}'
assert rec_hard > 0.15, '部分真难例应被识别为 hard 而不是 mislabeled'
print('✅ 练习 3 通过：**轨迹（AUM）给可疑度，一致性给「是错还是难」** —— 两个维度缺一不可')

## ✏️ 练习 4：安全挖掘比例与总放大倍数封顶

实现 `mining_plan(q, A, q_max, repeat_factor, amp_cap)`，返回 dict：

- `'r'`：安全挖掘比例（第 8 节的公式，结果裁剪到 `[0, 1]`；`A<=1` 时为 1.0；`q>=q_max` 时为 0.0）
- `'total_amp'`：稀有类样本的**总放大倍数** = `repeat_factor × (1 + r*(A-1))`
- `'ok'`：`total_amp <= amp_cap` 时为 True

（`1 + r(A-1)` 就是「一个噪声/难样本在混合批里相对原始频率的期望放大倍数」。）

In [ ]:
def mining_plan(q, A, q_max, repeat_factor=1.0, amp_cap=10.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
p1 = mining_plan(0.08, 6.0, 0.15)
assert abs(p1['r'] - 0.175) < 1e-9
assert abs(p1['total_amp'] - (1 + 0.175 * 5)) < 1e-9 and p1['ok']
assert mining_plan(0.15, 6.0, 0.15)['r'] == 0.0
assert mining_plan(0.02, 1.0, 0.15)['r'] == 1.0
# 与模块 01 的 repeat factor sampling 叠加 -> 总放大倍数可能爆掉
p2 = mining_plan(0.02, 8.0, 0.20, repeat_factor=6.0, amp_cap=10.0)
assert not p2['ok'], '重采样 6× 叠加激进挖掘应触发封顶告警'
print(f'{"场景":<34s}{"r":>8s}{"总放大":>10s}{"ok":>6s}')
for name, kw in [('干净数据、无重采样', dict(q=0.02, A=6.0, q_max=0.15)),
                 ('脏数据、无重采样', dict(q=0.08, A=6.0, q_max=0.15)),
                 ('干净数据 + 稀有类 6× 重采样', dict(q=0.02, A=8.0, q_max=0.20, repeat_factor=6.0)),
                 ('脏数据 + 稀有类 6× 重采样', dict(q=0.08, A=8.0, q_max=0.15, repeat_factor=6.0))]:
    pl = mining_plan(**kw)
    print(f'{name:<32s}{pl["r"]:>8.1%}{pl["total_amp"]:>10.2f}{("✅" if pl["ok"] else "❌"):>6s}')
print('✅ 练习 4 通过：**长尾重采样 × 难例挖掘会互相放大** —— 要给「总放大倍数」显式封顶')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def ohem_select(boxes, losses, k, iou_thr=0.7, min_loss=0.0):
    cand = np.where(losses >= min_loss)[0]              # ① 过滤简单样本
    if cand.size == 0:
        return cand
    keep_local = nms(boxes[cand], losses[cand], iou_thr) # ② 用 loss 当分数做 NMS
    return cand[keep_local][:k]                          # ③ 取 top-k（nms 已按 loss 降序）

In [ ]:
# 练习 2 参考答案
def focal_weights(p_t, gamma, alpha=None):
    w = (1.0 - np.asarray(p_t, dtype=float)) ** gamma
    return w if alpha is None else alpha * w

def equivalent_k(p_t, gamma):
    w = focal_weights(p_t, gamma)
    return int(round(w.sum() ** 2 / (w ** 2).sum()))

In [ ]:
# 练习 3 参考答案
def triage(margins, votes, labels, susp_pct=10.0, consensus_thr=0.8):
    aum_ = np.asarray(margins, dtype=float).mean(axis=1)
    thr = np.percentile(aum_, susp_pct)
    n_cls = int(max(votes.max(), labels.max())) + 1
    out = np.full(len(aum_), 'ok', dtype=object)
    for i in range(len(aum_)):
        if aum_[i] > thr:
            continue
        c = np.bincount(votes[i], minlength=n_cls)
        maj, frac = int(c.argmax()), c.max() / votes.shape[1]
        if maj != labels[i] and frac >= consensus_thr:
            out[i] = 'mislabeled'          # 模型一致地指向另一个类 -> 强烈怀疑标错
        elif frac < consensus_thr:
            out[i] = 'hard'                # 模型互相吵架 -> 真难例
    return out

In [ ]:
# 练习 4 参考答案
def mining_plan(q, A, q_max, repeat_factor=1.0, amp_cap=10.0):
    if A <= 1.0:
        r = 1.0
    elif q >= q_max:
        r = 0.0
    else:
        r = min(1.0, max(0.0, (q_max - q) / (q * (A - 1.0))))
    total = repeat_factor * (1.0 + r * (A - 1.0))
    return {'r': r, 'total_amp': total, 'ok': total <= amp_cap}

---
## 🧪 真实工程胶囊：一份可直接抄进项目的难例挖掘配置与自检清单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# 难例挖掘 · 生产配置模板（mmdet / 自研训练框架通用）
# ══════════════════════════════════════════════════════════════════

# ── ① 选 OHEM 还是 Focal ───────────────────────────────────────────
#  密集单阶段检测器（YOLO / RetinaNet / RTMDet） -> **Focal**（无需排序，可向量化）
#  两阶段第二阶段（Fast R-CNN head）              -> **OHEM**（RPN 已粗筛，focal 收益很小）
#  ⚠️ 两者**不要同时开**：等效样本量被平方级压缩，训练必然不稳。

sampler = dict(
    type="OHEMSampler",
    num=512, pos_fraction=0.25,      # ② **正负分开挖**，维持 1:3
    dedup_nms_iou=0.7,               # ③ **先用 loss 当分数做 NMS —— 缺这行 OHEM 会失效**
    loss_normalizer="num_selected",  # ④ 除以 k 而不是 N（否则等于偷偷降 lr）
    warmup_iters=1000,               # ⑤ warmup 期均匀采样（初期 loss 排序 = 随机排序）
)
loss_cls = dict(type="FocalLoss", gamma=2.0, alpha=0.25)
#   γ=2 在典型分布下 ≈ 只让 20% 左右的样本真正贡献梯度（用 n_eff=(Σw)²/Σw² 核算）
#   α 只平衡前景-背景，**不解决类别长尾**（那是 EQL / CB-Loss / logit adjustment 的活）

# ── ⑥ 挖掘比例的安全上界（由标注质量决定，不是拍脑袋）────────────────
#   q      = 标注噪声率（**必须靠重标注抽检得到**，~865 样本可估到 ±2%）
#   A      = 噪声放大系数 = P(noisy|selected)/P(noisy)（在自己数据上实测，常见 4–10）
#   r <= (q_max - q) / (q * (A - 1))
#   例: q=8%, A=6, q_max=15%  ->  r <= 17.5%   （**不是 100%**）

# ── ⑦ 训练时**免费**记录 AUM（强烈建议默认开启）──────────────────────
#   每个 epoch 存一次 per-sample margin = z[y] - max(z[c!=y])
#   100 万样本 × 50 epoch × float32 = 200 MB
#   AUM 最低 2–5% 进「可疑池」-> 用多模型一致性二次判别 -> 抽样重标验收
aum_log = dict(enable=True, every_n_epochs=1, out="aum_margins.npy")

# ── ⑧ 可疑样本：**降权而不是删除** ─────────────────────────────────
#   删除的风险是删掉真难例（最宝贵）且悄悄改变数据分布
per_sample_weight = dict(suspicious=0.2, normal=1.0)

# ── ⑨ 难负样本入库前的「可分性检查」───────────────────────────────
#   把 crop 单独给人看：**仅凭这块图像**能否判断它不是标志？
#   不能 -> 它不该做检测端的难负样本，而应作为下游（车道关联/跟踪）的测试用例
#   ⚠️ TSR 里最致命的错误：把**漏标的真标志**当难负样本 = 主动教模型漏检

# ── ⑩ 与长尾重采样的交互：给总放大倍数封顶 ───────────────────────────
#   total_amp = repeat_factor(类别) × (1 + r*(A-1))   <= 10
#   症状：稀有类训练 loss 极低但验证召回不涨 = 纯粹过拟合了那几十个样本

# ── 上线前自检清单 ────────────────────────────────────────────────
#   [ ] OHEM 里有 NMS 去重吗？（打印 top-k 覆盖的不同区域数，应 >= 0.5*k）
#   [ ] 正负分开挖了吗？（打印 top-k 里的正样本数，不能为 0）
#   [ ] warmup 期关掉挖掘了吗？
#   [ ] 噪声率 q 是**测出来的**还是猜的？
#   [ ] 挖掘比例 r 满足安全上界吗？
#   [ ] AUM 记录开了吗？可疑池是降权还是删除？
#   [ ] 难负样本库里有没有混进漏标的真目标？（抽 100 个人工过一遍）
'''
print(RECIPE)
for key in ['dedup_nms_iou', 'pos_fraction', 'num_selected', 'warmup_iters',
            'FocalLoss', 'aum_log', 'suspicious', 'total_amp', '可分性检查']:
    assert key in RECIPE, key
print('✅ 配方覆盖：算法选型 / 四个实现细节 / 安全比例 / AUM / 降权 / 可分性检查 / 总放大封顶')

### 小结

- **难例挖掘解决的是「梯度预算分配」**：98% 的 anchor 梯度只有 1e-3 量级，一个 s=0.8 的难负样本
  抵得上 800 个简单负样本。三代方法（bootstrapping → OHEM → Focal）是同一思想在数据/采样/损失
  三个层面的实现。
- **OHEM 的成败在「先用 loss 当分数做一遍 NMS」**：不去重时 128 个样本其实是 2–4 件事的拷贝，
  有效 batch ≈ 3，**比不做 OHEM 更差**。另外三条细节：正负分开挖、除以 k 而不是 N、warmup 期不挖。
- **Focal Loss 是软性 OHEM**：都能写成 `Σ w(ℓ)·ℓ`；**排序完全一致**（focal 权重与 CE 都是 p_t 的
  单调减函数）；差别只在被排除样本的权重是 0 还是 ε。用 `n_eff=(Σw)²/Σw²` 可把 γ 翻译成等价的 k。
  **两阶段检测器上 focal 收益小，因为 RPN 已经把前景背景比粗筛到 1:3。**
- **难例有四类，只有前三类值得挖**：相似类混淆 / 背景误检 / 边界样本 / **标注错误**。
  而 loss 排序把它们混在一起，且**挖得越狠，第四类占比越高**（噪声放大系数 A 常在 4–10）。
- **挖掘过头会毁掉模型**：噪声样本的 loss 永远最高 → 每轮都被选中 → 梯度预算被它们吃掉 →
  加速记忆化 → 泛化崩坏。而验证集同源同噪，**离线指标看不出来**。
  安全上界：`r ≤ (q_max − q) / (q(A−1))`。
- **区分难例与噪声的三件套**：**AUM/损失轨迹**（几乎零成本，默认开；只看最终 loss 分不开，
  看整条轨迹分得开）+ **多模型一致性**（一致地指向另一个类 = 疑似标错；互相吵架 = 真难例）
  + **重标注抽检**（~865 样本可把噪声率估到 ±2%，这是唯一的真值来源）。
- **可疑样本默认降权而不是删除**，因为误删的可能是最宝贵的真难例；也因为降权可逆、不改变分布。
- **TSR 落地**：难负样本要过「可分性检查」；**漏标的真标志伪装成 FP** 是最致命的陷阱
  （当负样本训练 = 主动教模型漏检）；难例池要按失效模式分桶并给每桶配额。

下一站：**模块 03 · 主动学习与线上触发策略** —— 车队数据无限、标注预算有限，
问题从「怎么挖」变成「**回传什么**」。